# 🔧 KEIBA-AI チューニングノート v2

**実行タイミング**：月1〜2回（新レースデータが50件以上増えたとき）

| セル | 内容 |
|---|---|
| セル1 | セットアップ |
| セル2 | 強制アップデート |
| セル3 | エンジン初期化 |
| セル4 | **重みチューニング** |
| セル5 | **キャリブレーション** |
| セル6 | 乖離分析 |
| セル7 | 反映確認 |


In [ ]:
# ── セル1: セットアップ ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json

BASE_DIR = '/content/drive/MyDrive/keiba_ai'
os.makedirs(f'{BASE_DIR}/data', exist_ok=True)

from datetime import datetime, timezone, timedelta
JST = timezone(timedelta(hours=9))
print(f'✅ セットアップ完了 ({datetime.now(JST).strftime("%Y-%m-%d %H:%M:%S")} JST)')
print(f'BASE_DIR: {BASE_DIR}')


In [ ]:
# ── セル2: 強制アップデート（毎回実行） ─────────────────────────────
import urllib.request

BASE_URL = 'https://raw.githubusercontent.com/hanagenuku/keiba_ai/main'
SRC_FILES = [
    'src/__init__.py',
    'src/tools/__init__.py',
    'src/tools/tune_weights.py',
    'src/tools/calibrate.py',
    'src/tools/analyze_divergence.py',
    'src/tools/rescrape_history.py',
    'src/tools/bias.py',
    'src/features/__init__.py',
    'src/features/engine.py',
    'src/utils/__init__.py',
    'src/utils/config.py',
    'src/utils/db.py',
    'src/utils/model_registry.py',
    'src/utils/jst.py',
    'src/scraper/__init__.py',
    'src/scraper/parser.py',
    'src/scraper/jra_scraper.py',
    'src/scraper/calendar.py',
    'src/models/__init__.py',
    'src/models/calibration.py',
    'src/models/predict.py',
    'src/betting/__init__.py',
    'src/betting/make_bets.py',
    'src/betting/ev_filter.py',
    'src/betting/app_json.py',
]
for rel in SRC_FILES:
    dest = f'{BASE_DIR}/{rel}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    urllib.request.urlretrieve(f'{BASE_URL}/{rel}', dest)
    print(f'✅ {rel}')
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]
sys.path.insert(0, BASE_DIR)
print('🔄 アップデート完了')


In [ ]:
# ── セル3: エンジン初期化 ────────────────────────────────────────────
from src.features.engine import init_engine
init_engine(BASE_DIR)
print('✅ エンジン初期化完了')


In [ ]:
# ── セル4: 重みチューニング ──────────────────────────────────────────
# 所要時間: 1〜3分（n_restarts を増やすと精度↑・時間↑）
from src.tools.tune_weights import run_tuning

opt_weights = run_tuning(
    BASE_DIR,
    n_restarts=5,  # 多いほど精度↑
    verbose=True,
)


In [ ]:
# ── セル5: キャリブレーション ────────────────────────────────────────
# ⚠ セル4（重みチューニング）の後に実行してください
from src.tools.calibrate import run_calibration

calibrator = run_calibration(
    BASE_DIR,
    method='isotonic',  # 'isotonic'（推奨）or 'platt'
    verbose=True,
)


In [ ]:
# ── セル6: 乖離分析・診断レポート ───────────────────────────────────
from src.tools.analyze_divergence import run_analysis
stats = run_analysis(BASE_DIR)


In [ ]:
# ── セル7: 反映確認 ──────────────────────────────────────────────────
from src.features.engine import init_engine
init_engine(BASE_DIR)

import json, os
w_path = f'{BASE_DIR}/data/optimal_weights.json'
c_path = f'{BASE_DIR}/data/calibrator.pkl'

print('\n【最適重み】')
if os.path.exists(w_path):
    with open(w_path) as f:
        w = json.load(f)
    total = sum(w.values())
    for k, v in sorted(w.items(), key=lambda x: -x[1]):
        bar = '█' * int(v * 100)
        print(f'  {k:12s}: {v:.4f}  {bar}')
    print(f'  合計: {total:.4f}')
else:
    print('  ⚠ optimal_weights.json が見つかりません')

print(f'\n【キャリブレーター】{"✅ 存在" if os.path.exists(c_path) else "❌ なし"}')
print('\n✅ 次回の金曜/土日ノート実行時に自動適用されます')
